# Tutorial 3: Compute persistence images in multiple directories

This note book illustrates how one computes persistence images, which are vectoizations of persistence diagrams.

In [1]:
using Pkg
Pkg.activate("../.")
Pkg.instantiate()

  Activating project at `~/Documents/TDA_on_lung_ECM`


In [2]:
include("../src/ECM_TDA.jl")

using .ECM_TDA
using PersistenceDiagrams
using DataFrames
using CSV
using DelimitedFiles
using Ripserer

# Compute persistence images for persistence diagrams in a single directory
* In most cases, you won't need to compute this separately. The persistence images are automatically computed when you run any of the following scripts:
    - `compute_persistence.jl`
    - `compute_persistence_script.jl`
    - `compute_Dowker_persistence.jl`
    - `compute_Dowker_persistence_script.jl`


In [3]:
# load persistence PersistenceDiagrams in a directory, organize them in a dictionary

PDs = Dict()
PD_dir = "PH_outputs/PD/PD1"
for file in readdir(PD_dir)
    filepath = joinpath(PD_dir, file)
    df = CSV.read(filepath, DataFrame; delim = ',', header = [:x, :y])
    arr = Matrix(df)
    PDs[file] = arr
end


Here is what the dictionary looks like

In [4]:
PDs

Dict{Any, Any} with 8 entries:
  "ROI4.csv" => [31.3209 31.7805; 33.0151 33.541; … ; 1139.15 1517.75; 571.316 …
  "ROI1.csv" => [56.0803 56.1427; 119.817 119.942; … ; 145.454 386.29; 585.03 8…
  "ROI6.csv" => [129.016 129.074; 66.1891 66.2722; … ; 261.605 709.677; 248.823…
  "ROI7.csv" => [62.0 62.0081; 37.0 37.0135; … ; 520.332 754.718; 675.41 1025.6…
  "ROI5.csv" => [59.4811 60.2163; 183.576 184.391; … ; 542.322 926.096; 308.084…
  "ROI8.csv" => [41.6773 41.7732; 86.977 87.0919; … ; 136.693 296.84; 162.25 15…
  "ROI2.csv" => [38.9487 39.0128; 84.2021 84.3445; … ; 260.409 486.199; 560.012…
  "ROI3.csv" => [79.2591 79.3095; 336.657 337.013; … ; 436.139 656.415; 1090.0 …

The following cell computes the persistence images

In [5]:
# convert array to Ripserer PD
PH = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PDs if v != nothing)

# compute PI
PersIm = PersistenceImage([PH[k] for k in keys(PH)], sigma=50, size = 20)

PI = Dict()
for i in keys(PH)
    PI[i] = PersIm(PH[i])
end


# Compute persistences images in multiple directories

Sometimes, we'll want to compare the topological features from two different directories. In this case, we want to make sure that the persistence images are computed at a comparable scale. One way to do this is to compute the persistence image in one directory, save the scale parameters involved, and then input the scale parameters when computing the persistence image in the second directory.

In [15]:
# compute persistence image from one of the directories
PD_dir1 = Dict()
PD_dir = "PI_tutorial/PD1_dir1"
for file in readdir(PD_dir)
    if file != ".DS_Store"
        filepath = joinpath(PD_dir, file)
        df = CSV.read(filepath, DataFrame; delim = ',', header = [:x, :y])
        arr = Matrix(df)
        PD_dir1[file] = arr
    end
end

# convert array to Ripserer PD
PH_dir1 = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PD_dir1 if v != nothing)


# compute PI
PersIm_dir1 = PersistenceImage([PH_dir1[k] for k in keys(PH_dir1)], sigma=50, size = 20)

PI_dir1 = Dict()
for i in keys(PH_dir1)
    PI_dir1[i] = PersIm_dir1(PH_dir1[i])
end


Save the parameters involved

In [16]:
# get the parameters involved 
PI_xmin = PersIm_dir1.xs[1]
PI_xmax = PersIm_dir1.xs[end]
PI_ymin = PersIm_dir1.ys[1]
PI_ymax = PersIm_dir1.ys[end];

Now compute the persistence images for the persistence diagrams in the second directory.

In [17]:
# compute persistence image from one of the directories
PD_dir2 = Dict()
PD_dir = "PI_tutorial/PD1_dir2"
for file in readdir(PD_dir)
    if file != ".DS_Store"
        filepath = joinpath(PD_dir, file)
        df = CSV.read(filepath, DataFrame; delim = ',', header = [:x, :y])
        arr = Matrix(df)
        PD_dir2[file] = arr
    end
end

# convert array to Ripserer PD
PH_dir2 = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PD_dir2 if v != nothing)

Dict{String, PersistenceDiagram} with 4 entries:
  "ROI6.csv" => 179-element PersistenceDiagram
  "ROI7.csv" => 263-element PersistenceDiagram
  "ROI5.csv" => 54-element PersistenceDiagram
  "ROI8.csv" => 286-element PersistenceDiagram

This time, when we define persistence image, we make sure to pass the parameters

In [18]:
PersIm_dir2 = PersistenceImage((PI_ymin, PI_ymax),(PI_xmin, PI_xmax), sigma= 50, size = 20)

PI_dir2 = Dict()
for i in keys(PH_dir2)
    PI_dir2[i] = PersIm_dir2(PH_dir2[i])
end